# Access the virtual Icechunk Repo FMRC using Rolodex 
This SHYFEM coastal model runs and pushes netCDF files to object storage each day, and creating references in virtualizarr2 to update the repo. This notebook opens the up-to-date forecast model run collection and uses Rolodex to extract BestEstimate time series

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
# load AWS credentials for Pangeo-EOSC storage as environment vars
import os
import icechunk
import xarray as xr
from obstore.store import S3Store
from dotenv import load_dotenv
# Load credentials
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env')

In [4]:
# Define storage
storage_endpoint = os.environ['ENDPOINT_URL']
storage_bucket = 'protocoast-data'
storage_name = 'taranto-icechunk-tubitak-v1'

In [5]:
store = S3Store(
    bucket="protocoast-data",  # or parse from your bucket_url
    endpoint=storage_endpoint, # e.g., "https://rustfs.vm.fedcloud.eu:9001"
    region="not-used",
)


In [6]:
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',   # N/A for Pangeo-EOSC bucket, but required param
    force_path_style=True)

In [7]:
config = icechunk.RepositoryConfig.default()

config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"s3://{storage_bucket}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True, 
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)


In [8]:
credentials = icechunk.containers_credentials({f"s3://{storage_bucket}/": icechunk.s3_credentials(anonymous=False)})

read_repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)

read_session = read_repo.readonly_session("main")

In [10]:
ds = xr.open_zarr(read_session.store, consolidated=False, zarr_format=3, chunks={})
ds['salinity']

<xarray.DataArray 'salinity' (time: 202, step: 144, node: 30731, level: 70)> Size: 250GB
dask.array<open_dataset-salinity, shape=(202, 144, 30731, 70), dtype=float32, chunksize=(1, 72, 16000, 1), chunktype=numpy.ndarray>
Coordinates:
  * level        (level) float32 280B 2.0 4.0 6.0 8.0 ... 1.5e+03 2e+03 2.5e+03
  * step         (step) timedelta64[ns] 1kB 00:00:00 ... 5 days 23:00:00
  * time         (time) datetime64[ns] 2kB 2025-09-08 2025-09-09 ... 2026-04-19
    longitude    (node) float32 123kB dask.array<chunksize=(16000,), meta=np.ndarray>
    topology     int32 4B ...
    latitude     (node) float32 123kB dask.array<chunksize=(16000,), meta=np.ndarray>
    total_depth  (node) float32 123kB dask.array<chunksize=(16000,), meta=np.ndarray>
Dimensions without coordinates: node
Attributes:
    units:          1e-3
    standard_name:  sea_water_salinity
    valid_min:      0.0
    valid_max:      200.0
    mesh:           mesh_topology
    location:       node

In [ ]:
import rolodex.forecast
from rolodex.forecast import (
    BestEstimate,
    ConstantForecast,
    ConstantOffset,
    ForecastIndex,
    Model,
    ModelRun,
)

In [ ]:
ds.coords["valid_time"] = rolodex.forecast.create_lazy_valid_time_variable(
    reference_time=ds.time, period=ds.step
)

In [ ]:
newds = ds.drop_indexes(["time", "step"]).set_xindex(
    ["time", "step", "valid_time"], ForecastIndex)

In [ ]:
newds

In [ ]:
ds_best = newds.sel(valid_time=BestEstimate(offset=2))  # start at forecast hour 2 instead of 0 (analysis time)

In [ ]:
ds_best

In [ ]:
import hvplot.xarray

In [ ]:
%%time
da = ds_best['temperature'][:,100,0].compute()

In [ ]:
da.hvplot(x='valid_time', grid=True)